# Tokyo: CLIP Embeddings + OSM Category + Reverse Geocoding\n
\n
This notebook reads the Tokyo OK dataset CSV, computes CLIP 512-D image embeddings, reverse-geocodes each (lat, lon) to get a place name, derives an OSM functional taxonomy category (same logic as `L1_1_vs_L1_2_Comparison.ipynb`), and writes `Other/Dataset/cluster_testing.csv`.

In [1]:
import json
import sqlite3
import time
import urllib.parse
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

pd.set_option("display.max_columns", 50)

In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "Other" / "Dataset" / "tokyo_first100_local_ok.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find the DREAMS repo root from the current working directory.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
TOKYO_OK_CSV = REPO_ROOT / "Other" / "Dataset" / "tokyo_first100_local_ok.csv"
TOKYO_IMAGE_DIR = REPO_ROOT / "Other" / "Dataset" / "tokoyo_images"
CACHE_DB_PATH = REPO_ROOT / "analysis_pipeline" / "data" / "cache" / "geocode_cache.db"
OUT_CSV = REPO_ROOT / "Other" / "Dataset" / "cluster_testing.csv"

ENABLE_LIVE_GEOCODING = True
GEOCODE_SLEEP_SECONDS = 1.1

print("Repo root:", REPO_ROOT)
print("Input CSV:", TOKYO_OK_CSV)
print("Output CSV:", OUT_CSV)

Repo root: C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS
Input CSV: C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Other\Dataset\tokyo_first100_local_ok.csv
Output CSV: C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Other\Dataset\cluster_testing.csv


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model_id = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_model_id).to(device)
clip_processor = CLIPProcessor.from_pretrained(clip_model_id)
clip_model.eval()

print(f"Using device: {device}")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Using device: cpu


In [4]:
_MEANINGLESS_PLACE_TYPES = {
    "post_box", "postbox", "street", "road", "path", "footway",
    "cycleway", "service", "bench", "waste_basket", "bollard",
    "fire_hydrant", "manhole", "drain", "tree", "street_lamp",
    "telephone", "atm", "vending_machine", "parking", "parking_space",
    "building", "yes", "construction", "boundary", "administrative",
    "county", "state", "country", "neighbourhood", "suburb",
    "quarter", "city", "town", "hamlet", "board",
}

_PLACE_CATEGORY_MAP = {
    "church": "faith_space",
    "cathedral": "faith_space",
    "chapel": "faith_space",
    "mosque": "faith_space",
    "temple": "faith_space",
    "synagogue": "faith_space",
    "place_of_worship": "faith_space",
    "shrine": "faith_space",
    "wayside_shrine": "faith_space",
    "park": "outdoor_nature",
    "garden": "outdoor_nature",
    "nature_reserve": "outdoor_nature",
    "forest": "outdoor_nature",
    "beach": "outdoor_nature",
    "trail": "outdoor_nature",
    "recreation_ground": "outdoor_nature",
    "national_park": "outdoor_nature",
    "playground": "outdoor_nature",
    "grassland": "outdoor_nature",
    "wetland": "outdoor_nature",
    "water": "outdoor_nature",
    "hospital": "clinical_setting",
    "clinic": "clinical_setting",
    "pharmacy": "clinical_setting",
    "doctors": "clinical_setting",
    "dentist": "clinical_setting",
    "health_centre": "clinical_setting",
    "nursing_home": "clinical_setting",
    "optician": "clinical_setting",
    "physiotherapist": "clinical_setting",
    "psychotherapist": "clinical_setting",
    "shelter": "support_service",
    "social_facility": "support_service",
    "community_centre": "support_service",
    "social_centre": "support_service",
    "food_bank": "support_service",
    "restaurant": "food_social",
    "cafe": "food_social",
    "fast_food": "food_social",
    "bar": "food_social",
    "pub": "food_social",
    "food_court": "food_social",
    "ice_cream": "food_social",
    "biergarten": "food_social",
    "bakery": "food_social",
    "school": "education",
    "university": "education",
    "college": "education",
    "library": "education",
    "kindergarten": "education",
    "language_school": "education",
    "driving_school": "education",
    "office": "work_space",
    "workplace": "work_space",
    "commercial": "work_space",
    "industrial": "work_space",
    "gym": "fitness_sport",
    "sports_centre": "fitness_sport",
    "swimming_pool": "fitness_sport",
    "stadium": "fitness_sport",
    "fitness_centre": "fitness_sport",
    "pitch": "fitness_sport",
    "track": "fitness_sport",
    "sports_hall": "fitness_sport",
    "bus_stop": "transit",
    "bus_station": "transit",
    "train_station": "transit",
    "station": "transit",
    "subway_entrance": "transit",
    "ferry_terminal": "transit",
    "airport": "transit",
    "halt": "transit",
    "tram_stop": "transit",
    "taxi": "transit",
    "supermarket": "retail",
    "convenience": "retail",
    "department_store": "retail",
    "mall": "retail",
    "marketplace": "retail",
    "clothes": "retail",
    "electronics": "retail",
    "hardware": "retail",
    "house": "residential",
    "apartments": "residential",
    "dormitory": "residential",
    "detached": "residential",
}

_CAPTION_KEYWORD_MAP = {
    "church": "faith_space",
    "chapel": "faith_space",
    "cathedral": "faith_space",
    "mosque": "faith_space",
    "temple": "faith_space",
    "synagogue": "faith_space",
    "park": "outdoor_nature",
    "garden": "outdoor_nature",
    "trail": "outdoor_nature",
    "beach": "outdoor_nature",
    "forest": "outdoor_nature",
    "lake": "outdoor_nature",
    "river": "outdoor_nature",
    "mountain": "outdoor_nature",
    "hospital": "clinical_setting",
    "clinic": "clinical_setting",
    "pharmacy": "clinical_setting",
    "doctor": "clinical_setting",
    "therapy": "clinical_setting",
    "shelter": "support_service",
    "food bank": "support_service",
    "community center": "support_service",
    "community centre": "support_service",
    "restaurant": "food_social",
    "cafe": "food_social",
    "coffee": "food_social",
    "bakery": "food_social",
    "school": "education",
    "university": "education",
    "college": "education",
    "library": "education",
    "gym": "fitness_sport",
    "workout": "fitness_sport",
    "fitness": "fitness_sport",
    "bus stop": "transit",
    "train station": "transit",
    "station": "transit",
    "airport": "transit",
    "store": "retail",
    "shop": "retail",
    "supermarket": "retail",
    "mall": "retail",
    "home": "residential",
    "house": "residential",
    "apartment": "residential",
}

def ensure_geocode_cache() -> sqlite3.Connection:
    CACHE_DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(CACHE_DB_PATH)
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS geocode_cache (
            lat_round REAL NOT NULL,
            lon_round REAL NOT NULL,
            response TEXT NOT NULL,
            cached_at TEXT NOT NULL DEFAULT (datetime('now')),
            PRIMARY KEY (lat_round, lon_round)
        )
        """
    )
    conn.commit()
    return conn

cache_conn = ensure_geocode_cache()
_last_geocode_at = None

def get_cached_geocode(lat: float, lon: float, precision: int = 4):
    lat_r = round(float(lat), precision)
    lon_r = round(float(lon), precision)
    row = cache_conn.execute(
        'SELECT response FROM geocode_cache WHERE lat_round = ? AND lon_round = ?',
        (lat_r, lon_r),
    ).fetchone()
    if row is None:
        return None
    return json.loads(row[0])

def set_cached_geocode(lat: float, lon: float, response: dict, precision: int = 4):
    lat_r = round(float(lat), precision)
    lon_r = round(float(lon), precision)
    cache_conn.execute(
        'INSERT OR REPLACE INTO geocode_cache (lat_round, lon_round, response) VALUES (?, ?, ?)',
        (lat_r, lon_r, json.dumps(response)),
    )
    cache_conn.commit()

def get_place_category(place_type: str):
    if not place_type:
        return None
    key = str(place_type).strip().lower()
    if not key or key in _MEANINGLESS_PLACE_TYPES:
        return None
    return _PLACE_CATEGORY_MAP.get(key)

def deep_parse_place_category(geo_data: dict):
    address = geo_data.get('address', {})
    for key in (
        'amenity', 'leisure', 'shop', 'tourism', 'building',
        'office', 'railway', 'aeroway', 'public_transport',
        'historic', 'landuse', 'natural'
    ):
        value = address.get(key, '')
        category = get_place_category(value)
        if category is not None:
            return category

    for raw in (geo_data.get('type'), geo_data.get('category')):
        category = get_place_category(raw)
        if category is not None:
            return category

    name = (geo_data.get('name') or geo_data.get('display_name') or '').lower()
    for keyword, category in sorted(_CAPTION_KEYWORD_MAP.items(), key=lambda item: len(item[0]), reverse=True):
        if keyword in name:
            return category
    return None

def caption_place_category(caption: str):
    text = (caption or '').lower()
    for keyword, category in sorted(_CAPTION_KEYWORD_MAP.items(), key=lambda item: len(item[0]), reverse=True):
        if keyword in text:
            return category
    return None

def extract_place_metadata(geo_data: dict, caption: str = '') -> dict:
    if not geo_data:
        return {'display_name': None, 'place_type': None, 'place_category': None}

    place_type = geo_data.get('type') or ''
    place_category = get_place_category(place_type)
    if place_category is None:
        place_category = deep_parse_place_category(geo_data)
    if place_category is None:
        place_category = caption_place_category(caption)

    display = geo_data.get('name') or geo_data.get('display_name')
    return {
        'display_name': display,
        'place_type': place_type or (geo_data.get('category') or None),
        'place_category': place_category,
    }

def reverse_geocode(lat: float, lon: float, allow_live: bool = True, retries: int = 3):
    global _last_geocode_at
    cached = get_cached_geocode(lat, lon)
    if cached is not None:
        return cached
    if not allow_live:
        return None

    params = urllib.parse.urlencode({
        'lat': float(lat),
        'lon': float(lon),
        'format': 'jsonv2',
        'addressdetails': 1,
    })
    url = f'https://nominatim.openstreetmap.org/reverse?{params}'
    headers = {'User-Agent': 'DREAMS-Cluster-Testing/0.1'}

    for attempt in range(retries):
        try:
            if _last_geocode_at is not None:
                elapsed = time.time() - _last_geocode_at
                if elapsed < GEOCODE_SLEEP_SECONDS:
                    time.sleep(GEOCODE_SLEEP_SECONDS - elapsed)
            request = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(request, timeout=20) as response:
                payload = json.load(response)
            _last_geocode_at = time.time()
            set_cached_geocode(lat, lon, payload)
            return payload
        except Exception:
            _last_geocode_at = time.time()
            time.sleep(min(2 ** attempt, 8))
    return None

def resolve_image_path(raw_path):
    if pd.isna(raw_path) or not str(raw_path).strip():
        return None
    path = Path(str(raw_path).strip())
    candidates = [path, TOKYO_IMAGE_DIR / path.name, REPO_ROOT / path.name]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

In [5]:
def clip_image_features(paths, batch_size: int = 8):
    features = [None] * len(paths)
    for start in range(0, len(paths), batch_size):
        batch_indices = []
        batch_images = []
        for idx in range(start, min(start + batch_size, len(paths))):
            path = paths[idx]
            if path is None:
                continue
            try:
                with Image.open(path) as image:
                    batch_images.append(image.convert('RGB'))
                batch_indices.append(idx)
            except Exception:
                continue
        if not batch_images:
            continue
        inputs = clip_processor(images=batch_images, return_tensors='pt', padding=True)
        inputs = {key: value.to(device) for key, value in inputs.items()}
        with torch.no_grad():
            emb = clip_model.get_image_features(**inputs)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        batch_array = emb.cpu().numpy()
        for offset, idx in enumerate(batch_indices):
            features[idx] = batch_array[offset]
    return features

In [6]:
df = pd.read_csv(TOKYO_OK_CSV).rename(columns={'photo/video_page_url': 'image_path'})
if 'latitude' not in df.columns or 'longitude' not in df.columns:
    raise KeyError('Expected latitude/longitude columns in the tokyo OK csv')
df['image_path'] = df['image_path'].astype(str).str.strip()
df['caption'] = df.get('caption', '').fillna('').astype(str)
print('Loaded rows:', len(df))
display(df.head())

Loaded rows: 159


,user_id,longitude,latitude,date_taken,x,y,image_path,caption,Time
0,10727420@N00,139.700499,35.674000,2010-04-09 17:26:25.0,1.555139e+07,4.255856e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,Today felt peaceful and steady.,2025-01-01T22:00:00
1,8819274@N04,139.766521,35.709095,2007-02-10 16:08:40.0,1.555874e+07,4.260667e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,Feeling happy and light today.,2025-01-02T15:50:00
2,62068690@N00,139.765632,35.694482,2008-12-21 15:45:31.0,1.555864e+07,4.258664e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,Feeling sad and missing home.,2025-01-04T09:10:00
3,49503094041@N01,139.784391,35.548589,2011-11-11 05:48:54.0,1.556073e+07,4.238684e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,"A quiet day, kind of neutral.",2025-01-05T14:05:00
4,40443199@N00,139.768753,35.671521,2006-04-06 16:42:49.0,1.555899e+07,4.255517e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,"A quiet day, kind of neutral.",2025-01-06T09:55:00


In [7]:
image_paths = [resolve_image_path(p) for p in df['image_path'].tolist()]
clip_feats = clip_image_features(image_paths, batch_size=8)

place_names = []
place_categories = []
place_types = []

for row in df.itertuples(index=False):
    geo = reverse_geocode(row.latitude, row.longitude, allow_live=ENABLE_LIVE_GEOCODING)
    meta = extract_place_metadata(geo, caption=getattr(row, 'caption', ''))
    place_names.append(meta.get('display_name'))
    place_types.append(meta.get('place_type'))
    place_categories.append(meta.get('place_category') or 'unknown_place')

emb_strings = []
missing_emb = 0
for feat in clip_feats:
    if feat is None:
        emb_strings.append('')
        missing_emb += 1
    else:
        emb_strings.append(json.dumps([float(x) for x in feat.tolist()]))

out = pd.DataFrame({
    'image_path': df['image_path'].tolist(),
    'latitude': df['latitude'].tolist(),
    'longitude': df['longitude'].tolist(),
    'place_name': place_names,
    'place_type': place_types,
    'clip_embedding_512': emb_strings,
    'category': place_categories,
})

valid_mask = out['category'].astype(str).str.strip().ne('unknown_place')
out_valid = out.loc[valid_mask].reset_index(drop=True)

print('Rows (total):', len(out))
print('Missing embeddings:', missing_emb)
print('Unknown categories:', int((out['category'] == 'unknown_place').sum()))
print('Rows kept (valid category only):', len(out_valid))
print('Unique categories kept:', sorted(out_valid['category'].unique().tolist()))

display(out_valid.head())

Rows (total): 159
Missing embeddings: 0
Unknown categories: 103
Rows kept (valid category only): 56
Unique categories kept: ['clinical_setting', 'education', 'faith_space', 'fitness_sport', 'food_social', 'outdoor_nature', 'residential', 'retail', 'transit', 'work_space']


,image_path,latitude,longitude,place_name,place_type,clip_embedding_512,category
0,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,35.694482,139.765632,神田キリスト教診療所,doctors,"[-0.04367450252175331, -0.001564520294778049, ...",clinical_setting
1,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,35.548589,139.784391,スターバックス,cafe,"[0.02233635075390339, 0.014968139119446278, 0....",food_social
2,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,35.669833,139.708000,Regatto Hat Shop,clothes,"[-0.008107353933155537, 0.03579247370362282, -...",retail
3,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,35.631244,139.797077,Tokyo Bay-kitchen,fast_food,"[0.006906784139573574, 0.0036685599479824305, ...",food_social
4,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,35.694353,139.703264,はじまりはいつも...ゴーロクキュー,pub,"[-0.009900021366775036, -0.0005098265246488154...",food_social


In [8]:
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
out_valid.to_csv(OUT_CSV, index=False)
print('Wrote (valid categories only):', OUT_CSV)

Wrote (valid categories only): C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Other\Dataset\cluster_testing.csv
